### ==============================================================================
## Processing of Moving Vessel Profiler Data - code 5_TEOS10_calculation_leg1
### Authors: Elisabet Verger-Miralles (everger@imedea.uib-csic.es)
### Data from FaSt-SWOT experiment (Leg 1)
# 
**DESCRIPTION**:
This script calculates standard TEOS-10 variables (Absolute Salinity, 
Conservative Temperature, Potential Density) using the GSW library from 
smoothed temperature and salinity data. It recalculates a consistent final 
conductivity and structures the NetCDF file with official standard names 
and CF-compliant metadata.
#
**NOTE**: No salinity offset is applied at this stage. The offset will be 
determined in Step 6 (via CTD comparison) and applied in Step 7.
#
INPUT: Smoothed NetCDF files (*_step4_loess.nc).
#
OUTPUT: Intermediate NetCDF datasets with TEOS-10 variables (*_step5.nc).
### ==============================================================================

In [1]:
import xarray as xr
import numpy as np
import gsw
from pathlib import Path



# ==========================================

# CONFIGURATION

# ==========================================

BASE_ROOT = Path(r"C:\Users\ASUS\Desktop\MVP\MVP_paper_data_figures_publish\MVP_fastswot_nc_final_processing")

DIR_IN = BASE_ROOT / "Data" / "Leg1" / "processed_step4_loess"

DIR_OUT = BASE_ROOT / "Data" / "Leg1" / "processed_step5"

DIR_OUT.mkdir(parents=True, exist_ok=True)


# ==========================================

# PROCESSING

# ==========================================

files = sorted(list(DIR_IN.glob("*_step4_loess.nc")))


for f in files:

    try:

        with xr.open_dataset(f) as ds:

            p = ds['pressure'].values
            t_smooth = ds['t1_smooth'].values
            s_smooth = ds['salinity_smooth'].values
            lat, lon = ds.latitude.values, ds.longitude.values

            # 1. Salinity Final (NO offset at this stage)

            s_final = s_smooth  # No offset applied here (step 5 is offset-free)

            # 2. Variables TEOS-10

            SA = gsw.SA_from_SP(s_final, p, lon, lat) # Absolute Salinity
            CT = gsw.CT_from_t(SA, t_smooth, p)      # Conservative Temperature

            sigma0 = gsw.sigma0(SA, CT)              # Potential Density
            

            # 3. FINAL CONDUCTIVITY (Consistent with the changes)

            # It is recalculated so that S_final = f(C_final, T_smooth, P)

            c_final = gsw.C_from_SP(s_final, t_smooth, p)


            # 4. Final Dataset Creation

            ds_final = ds.copy(deep=True)
            

            ds_final['temperature_conservative'] = (('scan',), CT)
            ds_final['salinity_practical'] = (('scan',), s_final)
            ds_final['conductivity_final'] = (('scan',), c_final)
            ds_final['sigma_theta'] = (('scan',), sigma0)


            # Metadata

            ds_final['temperature_conservative'].attrs = {'units': 'degC', 'standard_name': 'sea_water_conservative_temperature'}
            ds_final['salinity_practical'].attrs = {'units': 'PSU', 'standard_name': 'sea_water_practical_salinity', 'comment': 'No offset applied (step 5 - offset will be applied in step 7)'}
            ds_final['conductivity_final'].attrs = {'units': 'mS/cm', 'standard_name': 'sea_water_electrical_conductivity'}
            ds_final['sigma_theta'].attrs = {'units': 'kg/m^3', 'standard_name': 'sea_water_sigma_theta'}


            out_name = f.name.replace('_step4_loess.nc', '_step5.nc')  # No offset

            ds_final.to_netcdf(DIR_OUT / out_name)

            print(f"   ✅ Dataset created: {out_name}")

            

    except Exception as e:

        print(f"   ❌ Error in {f.name}: {e}")


print(f"\n Process completed. {len(files)} files saved in {DIR_OUT}")

c:\Users\ASUS\anaconda3\envs\env_elisabet\lib\site-packages\pandas\core\computation\expressions.py:21: UserWarning: Pandas requires version '2.8.4' or newer of 'numexpr' (version '2.7.3' currently installed).
  from pandas.core.computation.check import NUMEXPR_INSTALLED


   ✅ Dataset created: imedea-socib_fast-swot1_mvp_0009_step5.nc
   ✅ Dataset created: imedea-socib_fast-swot1_mvp_0011_step5.nc
   ✅ Dataset created: imedea-socib_fast-swot1_mvp_0013_step5.nc
   ✅ Dataset created: imedea-socib_fast-swot1_mvp_0015_step5.nc
   ✅ Dataset created: imedea-socib_fast-swot1_mvp_0019_step5.nc
   ✅ Dataset created: imedea-socib_fast-swot1_mvp_0021_step5.nc
   ✅ Dataset created: imedea-socib_fast-swot1_mvp_0023_step5.nc
   ✅ Dataset created: imedea-socib_fast-swot1_mvp_0025_step5.nc
   ✅ Dataset created: imedea-socib_fast-swot1_mvp_0026_step5.nc
   ✅ Dataset created: imedea-socib_fast-swot1_mvp_0028_step5.nc
   ✅ Dataset created: imedea-socib_fast-swot1_mvp_0033_step5.nc
   ✅ Dataset created: imedea-socib_fast-swot1_mvp_0037_step5.nc
   ✅ Dataset created: imedea-socib_fast-swot1_mvp_0041_step5.nc
   ✅ Dataset created: imedea-socib_fast-swot1_mvp_0042_step5.nc
   ✅ Dataset created: imedea-socib_fast-swot1_mvp_0043_step5.nc
   ✅ Dataset created: imedea-socib_fast-

In [2]:
# import matplotlib.pyplot as plt
# import xarray as xr
# import numpy as np
# import pandas as pd
# from pathlib import Path
# from geopy.distance import geodesic
# import gsw
# import re
# import io
# import warnings

# warnings.filterwarnings("ignore")

# # NOTE: This script generates pre-offset validation figures.
# # Final publication figures (with offset applied) are generated in Step 8.
# # Uncomment below to regenerate these pre-offset figures if needed.

# """
# # ==========================================
# # 1. CONFIGURATION
# # ==========================================
# BASE_ROOT = Path(r"C:\Users\ASUS\Desktop\MVP\MVP_paper_data_figures_publish\MVP_fastswot_nc_final_processing")

# DIR_MVP_RAW = BASE_ROOT / "Data" / "Leg1" / "processed_step1_highres_qc"
# DIR_MVP_FINAL = BASE_ROOT / "Data" / "Leg1" / "processed_step5_final" 
# DIR_CTD = Path(r"C:/Users/ASUS/OneDrive - Universitat de les Illes Balears/SWOT/CTD/CTD_data/leg1/HM/")

# OUTDIR = BASE_ROOT / "Figures" / "FIGURES_PAPER_FINAL_RESULTS"
# OUTDIR.mkdir(parents=True, exist_ok=True)

# # Parameters
# MAX_DIST_KM = 0.5    
# MAX_TIME_MIN = 60.0  
# Z_MAX_PLOT = 100.0   
# CTD_SMOOTH_WINDOW = 8 

# # ==========================================
# # 2. FUNCTIONS
# # ==========================================

# def read_ctd_robust(path):
#     try:
#         with open(path, 'r', encoding='latin-1') as f: lines = f.readlines()
#         lat, lon, time_val, start_idx = np.nan, np.nan, pd.NaT, 0
#         col_map = {}
#         for i, line in enumerate(lines):
#             if 'NMEA Latitude' in line:
#                 p = re.findall(r"[-+]?\d*\.\d+|\d+", line)
#                 if len(p)>=2: lat = float(p[0]) + float(p[1])/60 * (-1 if 'S' in line else 1)
#             if 'NMEA Longitude' in line:
#                 p = re.findall(r"[-+]?\d*\.\d+|\d+", line)
#                 if len(p)>=2: lon = float(p[0]) + float(p[1])/60 * (-1 if 'W' in line else 1)
#             if 'start_time' in line:
#                 try: time_val = pd.to_datetime(line.split('=')[1].strip().split('[')[0])
#                 except: pass
#             if '# name' in line:
#                 match = re.search(r"name (\d+)", line)
#                 if match:
#                     idx = int(match.group(1))
#                     name = line.split('=')[1].split(':')[0].strip().lower()
#                     # Lógica de coincidencia exacta para evitar columnas erróneas
#                     if name in ['prdm', 'pres', 'pressure']: col_map['p'] = idx
#                     elif name in ['t090c', 't190c', 'temp', 'tv290c']: col_map['t'] = idx
#                     elif name in ['sal00', 'sal11', 'sal', 'psal']: col_map['s'] = idx
#                     elif name in ['c0s/m', 'c1s/m', 'cond', 'c0ms/cm']: col_map['c'] = idx
#             if '*END*' in line: start_idx = i+1; break
        
#         df = pd.read_csv(io.StringIO("".join(lines[start_idx:])), delim_whitespace=True, header=None)
#         p, t, s = df.iloc[:, col_map['p']].values, df.iloc[:, col_map['t']].values, df.iloc[:, col_map['s']].values
#         c = df.iloc[:, col_map['c']].values if 'c' in col_map else None
#         if c is not None and np.nanmean(c) < 10: c *= 10.0
#         return {'lat': lat, 'lon': lon, 'time': time_val, 'id': path.stem, 'p': p, 't': t, 's': s, 'c': c}
#     except: return None

# def load_mvp_catalog(mvp_dir):
#     data = []
#     files = sorted(list(mvp_dir.glob("*.nc")))
#     for f in files:
#         try:
#             with xr.open_dataset(f) as ds:
#                 lat = ds.latitude.values.flatten()[0] if 'latitude' in ds else np.nan
#                 lon = ds.longitude.values.flatten()[0] if 'longitude' in ds else np.nan
#                 t_val = pd.to_datetime(ds.time.values.flatten()[0]) if 'time' in ds else pd.NaT
#                 if np.isfinite(lat):
#                     data.append({'file': f.name, 'lat': lat, 'lon': lon, 'time': t_val, 'path': f})
#         except: pass
#     return pd.DataFrame(data)

# # ==========================================
# # 3. PROCESSING AND PLOTTING
# # ==========================================

# print(f"Generating comparative figures with data from Step 5...")
# df_mvp = load_mvp_catalog(DIR_MVP_RAW)

# if df_mvp.empty:
#     print("❌ Error: No MVP data available.")
# else:
#     for ctd_p in sorted(DIR_CTD.glob("d*.cnv")):
#         ctd = read_ctd_robust(ctd_p)
#         if not ctd or pd.isna(ctd['time']): continue
        
#         df_mvp['dist_km'] = [geodesic((ctd['lat'], ctd['lon']), (m.lat, m.lon)).km for m in df_mvp.itertuples()]
#         df_mvp['dt_min'] = [(abs(m.time - ctd['time']).total_seconds() / 60.0) for m in df_mvp.itertuples()]
#         matches = df_mvp[(df_mvp['dist_km'] <= MAX_DIST_KM) & (df_mvp['dt_min'] <= MAX_TIME_MIN)].copy()
        
#         if matches.empty: continue
            
#         best_match = matches.loc[matches['dist_km'].idxmin()]
#         mvp_id = best_match['file'][27:33]
        
#         raw_path = DIR_MVP_RAW / best_match['file']
#         final_path = DIR_MVP_FINAL / best_match['file'].replace("_step1_qc.nc", "_step5_final.nc")

#         if not final_path.exists(): continue

#         try:
#             with xr.open_dataset(final_path) as ds_f, xr.open_dataset(raw_path) as ds_r:
#                 # --- LOAD OF VARIABLES STEP 5 (UPDATED NAMES) ---
#                 p_p = ds_f.pressure.values
#                 t_p = ds_f.temperature_conservative.values
#                 s_p = ds_f.salinity_practical.values
#                 c_p = ds_f.conductivity_final.values
                
#                 # Raw data
#                 p_r, t_r, c_r = ds_r.pressure.values, ds_r.t1.values, ds_r.c1.values
#                 if np.nanmean(c_r) < 10: c_r *= 10
#                 s_r = gsw.SP_from_C(c_r, t_r, p_r)

#             # --- CALCULATION OF METRICS ---
#             p_grid = np.arange(10, Z_MAX_PLOT, 1.0)
#             s_ctd_smooth = pd.Series(ctd['s']).rolling(window=CTD_SMOOTH_WINDOW, center=True, min_periods=1).mean().values
#             s_ctd_i = np.interp(p_grid, ctd['p'], s_ctd_smooth)
#             s_raw_i = np.interp(p_grid, p_r, s_r)
#             s_fin_i = np.interp(p_grid, p_p, s_p)
            
#             mask = np.isfinite(s_ctd_i) & np.isfinite(s_raw_i) & np.isfinite(s_fin_i)
#             rmsd_raw = np.sqrt(np.mean((s_ctd_i[mask] - s_raw_i[mask])**2))
#             rmsd_fin = np.sqrt(np.mean((s_ctd_i[mask] - s_fin_i[mask])**2))
#             imp = (1 - rmsd_fin/rmsd_raw) * 100

#             # --- PLOT ---
#             fig, axs = plt.subplots(1, 3, figsize=(16, 7), sharey=True, constrained_layout=True)
#             kw_raw = {'s': 5, 'color': '#CCCCCC', 'alpha': 0.7, 'label': 'MVP Raw'}
#             # LEGEND WITH % OF IMPROVEMENT
#             kw_fin = {'s': 4, 'color': 'red', 'alpha': 0.8, 'label': f'MVP Processed'}

#             # (A) Temperature
#             axs[0].scatter(t_r, p_r, **kw_raw)
#             axs[0].scatter(t_p, p_p, **kw_fin)
#             t_ctd_smooth = pd.Series(ctd['t']).rolling(window=CTD_SMOOTH_WINDOW, center=True, min_periods=1).mean().values
#             axs[0].plot(t_ctd_smooth, ctd['p'], color='black', lw=2, label='CTD Reference')
#             axs[0].set_xlabel("Temperature (°C)", fontsize=14)
#             axs[0].set_ylabel("Pressure (dbar)", fontsize=14)

#             tick_params = {'axis': 'both', 'which': 'both', 'labelsize': 14}
            
#             axs[0].tick_params(**tick_params)

#             # (B) Conductivity
#             axs[1].scatter(c_r, p_r, **kw_raw)
#             axs[1].scatter(c_p, p_p, **kw_fin)
#             if ctd['c'] is not None:
#                 c_ctd_smooth = pd.Series(ctd['c']).rolling(window=CTD_SMOOTH_WINDOW, center=True, min_periods=1).mean().values
#                 axs[1].plot(c_ctd_smooth, ctd['p'], color='black', lw=2)
#             axs[1].set_xlabel("Conductivity (mS/cm)", fontsize=14)
#             axs[1].tick_params(**tick_params)

#             # (C) Salinity
#             axs[2].scatter(s_r, p_r, **kw_raw)
#             axs[2].scatter(s_p, p_p, **kw_fin)
#             axs[2].plot(s_ctd_smooth, ctd['p'], color='black', lw=2.5, label='CTD Reference')
            
#             s_mid = np.nanmedian(s_p)
#             axs[2].set_xlim(s_mid - 0.4, s_mid + 0.4)
#             axs[2].set_xlabel("Practical Salinity", fontsize=14)
#             axs[2].legend(loc='upper right', fontsize=14)
#             axs[2].tick_params(**tick_params)

#             for ax in axs:
#                 ax.invert_yaxis(); ax.set_ylim(Z_MAX_PLOT, 0); ax.grid(True, alpha=0.3)

#             txt = (f"CTD: {ctd['id']}\nMVP: {mvp_id}\nDist: {best_match['dist_km']:.2f} km\nDt: {best_match['dt_min']:.1f} min\nErr. Red.: {imp:.1f}%")
#             axs[0].text(0.58, 0.05, txt, transform=axs[0].transAxes, fontsize=14, 
#                         bbox=dict(boxstyle='round', facecolor='white', alpha=0.9))

#             plt.savefig(OUTDIR / f"Fig_Final_{ctd['id']}.png", dpi=300)
#             plt.close()
#             print(f"   ✅ Station {ctd['id']} OK.")

#         except Exception as e:
#             print(f"   ❌ Error in {ctd['id']}: {e}")

# print(f"\n¡DONE! Figures generated in {OUTDIR}")
# """

# print("Step 5.2: Pre-offset figures generation is disabled."
#       "\nFinal publication figures are generated in Step 8 (with offset applied).")